### Game Genie Room Setup

Creates a Genie room for Level 1 of Casper's Kitchen Rescue.
The room is configured with the investigation views as trusted assets
and includes table-level instructions that guide discovery without
giving away answers.

In [ ]:
%pip install --upgrade databricks-sdk

In [ ]:
dbutils.library.restartPython()

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")

##### Create Genie Room via SDK

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Get a SQL warehouse for the Genie room
warehouses = list(w.warehouses.list())
warehouse_id = None
for wh in warehouses:
    if wh.name and CATALOG in wh.name:
        warehouse_id = wh.id
        break

# Fallback: use any available serverless warehouse
if not warehouse_id:
    for wh in warehouses:
        if wh.enable_serverless_compute or 'serverless' in (wh.name or '').lower():
            warehouse_id = wh.id
            break

# Last resort: use any warehouse
if not warehouse_id and warehouses:
    warehouse_id = warehouses[0].id

if not warehouse_id:
    raise RuntimeError("No SQL warehouse found. Create one first.")

print(f"Using warehouse_id={warehouse_id}")

In [ ]:
import json

table_identifiers = [
    f"{CATALOG}.game.investigation_revenue_daily",
    f"{CATALOG}.lakeflow.gold_order_header",
    f"{CATALOG}.simulator.locations",
]

serialized_space = json.dumps({
    "version": 2,
    "data_sources": {
        "tables": [{"identifier": t} for t in table_identifiers],
    },
})

description = """You are helping investigate revenue anomalies at Casper's Kitchens, a ghost kitchen chain.

The investigation_revenue_daily table tracks daily revenue by location. It has two revenue columns:
- revenue: the actual recorded revenue
- projected_revenue: what finance expected the revenue to be

Compare these across locations and over time to find patterns.

Tables available:
- investigation_revenue_daily: daily revenue with actual vs projected per location (columns: location_id, location_name, order_day, revenue, projected_revenue, orders)
- gold_order_header: per-order summary with location and revenue
- locations: location names and IDs

Use natural language questions to explore the data."""

print("Genie room configuration ready")
print(f"Tables: {table_identifiers}")

In [ ]:
import json, requests

genie_url = None
space_id = None
host = w.config.host.rstrip("/")
room_title = f"Casper's Kitchen Investigation ({CATALOG})"

try:
    token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
except Exception:
    token = w.config.token
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
print(f"Host: {host}")

# Check for existing room
try:
    list_resp = requests.get(f"{host}/api/2.0/genie/spaces", headers=headers)
    if list_resp.ok:
        for space in list_resp.json().get("spaces", []):
            if space.get("title") == room_title:
                space_id = space["space_id"]
                genie_url = f"{host}/genie/rooms/{space_id}"
                print(f"♻️ Found existing Genie room: {genie_url}")
                break
except Exception as e:
    print(f"  List failed: {e}")

if space_id:
    # Always update tables and description on existing rooms
    update_payload = {
        "title": room_title,
        "description": description,
        "warehouse_id": warehouse_id,
        "serialized_space": serialized_space,
    }
    try:
        resp = requests.patch(
            f"{host}/api/2.0/genie/spaces/{space_id}",
            headers=headers, json=update_payload,
        )
        if resp.ok:
            print(f"✅ Updated Genie room tables: {table_identifiers}")
        else:
            print(f"⚠️ Update returned {resp.status_code}: {resp.text[:300]}")
    except Exception as e:
        print(f"⚠️ Could not update room: {e}")
else:
    # Create new room
    try:
        genie_space = w.genie.create_space(
            warehouse_id=warehouse_id,
            serialized_space=serialized_space,
            title=room_title,
            description=description,
        )
        space_id = genie_space.space_id
        genie_url = f"{host}/genie/rooms/{space_id}"
        print(f"✅ Created Genie room via SDK: {genie_url}")
    except Exception as e:
        print(f"SDK create failed: {e}")
        payload = {
            "title": room_title,
            "description": description,
            "warehouse_id": warehouse_id,
            "serialized_space": serialized_space,
        }
        try:
            resp = requests.post(f"{host}/api/2.0/genie/spaces", headers=headers, json=payload)
            if resp.ok:
                data = resp.json()
                space_id = data.get("space_id") or data.get("id")
                if space_id:
                    genie_url = f"{host}/genie/rooms/{space_id}"
                    print(f"✅ Created Genie room: {genie_url}")
        except Exception as e2:
            print(f"REST create failed: {e2}")

if not genie_url:
    print("\n❌ Could not create Genie room.")
    print("Create one manually in the Databricks UI with these tables:")
    for t in table_identifiers:
        print(f"  - {t}")
else:
    print(f"\n✅ Genie room ready: {genie_url}")

In [ ]:
# Store the Genie room URL in the game config table and register with uc_state for cleanup
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.game.config (
  config_key STRING,
  config_value STRING
)
""")

if space_id:
    import sys
    sys.path.append('../utils')
    from uc_state import add
    add(CATALOG, "genie_spaces", {"space_id": space_id, "title": room_title})

if genie_url:
    spark.sql(f"""
    MERGE INTO {CATALOG}.game.config AS target
    USING (SELECT 'genie_room_url' AS config_key, '{genie_url}' AS config_value) AS source
    ON target.config_key = source.config_key
    WHEN MATCHED THEN UPDATE SET config_value = source.config_value
    WHEN NOT MATCHED THEN INSERT (config_key, config_value) VALUES (source.config_key, source.config_value)
    """)
    print(f"\u2705 Stored Genie room URL in game config")
else:
    print("\u26a0\ufe0f No Genie URL to store. Set it manually in game.config after creating the room.")